# NHS England Maternal Care Pathways Master Pipeline
## Stage 7.1 - Define pre-eclampsia indicators

### Compiled by Federica Caretta Cortegiani

In [0]:
import sys
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from collections import Counter 
from pyspark.sql.functions import when, col, lit, count, max, last, first, min, avg, sum, collect_list, collect_set, countDistinct, to_date, datediff, array_contains, size, udf, format_number, regexp_replace, explode, array, array_union, desc_nulls_last, flatten, split, filter
from pyspark.sql.types import IntegerType, StringType, NumericType, DateType, ArrayType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
from pyspark.ml.regression import GeneralizedLinearRegression as GLR

%run "/Workspace/Shared/Helper_Methods_EP"

In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
read_table_name = "msds_diag_busy_filtered_final_imputed_stage"

df_master = spark.read.table(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{read_table_name}")

In [0]:
df_master = (
    df_master
    .withColumns({
        "est_pregnancy_start": 
            when(col("est_delivery_date").isNotNull(), F.date_add(col("est_delivery_date"), -40*7))
            .otherwise(None),
        "est_ld_disch_date": 
            when(col("ld_hosp_disch_date").isNotNull(), F.date_add(col("ld_hosp_disch_date"), 2*7))
            .when(col("est_delivery_date").isNotNull(), F.date_add(col("est_delivery_date"), 2*7))
            .otherwise(None)
    })
)

df_master = (
    df_master
    .withColumn("gestation_length", F.date_diff("ld_hosp_disch_date", "last_period_date"))
    .withColumn(
        "ld_hosp_disch_date", when(((col("gestation_length") > 42*7) | (col("gestation_length") < 0)), F.date_add(col("last_period_date"), 42*7)).otherwise(col("ld_hosp_disch_date"))
    )
    .withColumn("est_gestation_length", F.date_diff("est_ld_disch_date", "est_pregnancy_start"))
    .withColumn(
        "est_ld_disch_date", when((col("est_gestation_length") > 42*7), F.date_add(col("est_pregnancy_start"), 42*7)).when((col("est_gestation_length") < 0), F.date_add(col("est_pregnancy_start"), 42*7)).otherwise(col("est_ld_disch_date"))
    )
    .drop(*["gestation_length", "est_gestation_length"])
)

In [0]:
w = Window.partitionBy("Person_ID_DEID").orderBy(F.col("last_period_date").asc())

df_reduced = (
    df_master
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

In [0]:
df_preg_ids = df_reduced.select("UniqPregID", "person_id_deid", "last_period_date", "ld_hosp_disch_date")


In [0]:
df_observations = spark.sql(
    "SELECT DISTINCT * FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.msds_v2_findings_and_observations_all_years"
)

df_msds_diags = spark.sql(
    "SELECT DISTINCT * FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.msds_v2_diagnoses_and_history_all_years "
)

df_hes_apc = spark.sql(
    "SELECT DISTINCT * FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.hes_apc_all_years "
)

df_ecds = spark.sql(
    "SELECT DISTINCT * FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.lowlat_ecds_all_years "
)

snomed_codes_desc = spark.read.table("reference_data.dss_corporate.snomed_ct_rf2_descriptions")
icd10_codes_desc = spark.read.table("reference_data.dss_corporate.icd10_codes")

In [0]:
df_msds_diags = (
    df_msds_diags
    .filter(col("DiagnosisType") == "Pregnancy")
)

df_observations = (
    df_observations
    .withColumn("DiagnosisDate", when(col("FindingDate").isNotNull(), col("FindingDate")).otherwise(col("ObsDate")))
)

In [0]:
def diagnosis_codes(desc_col, single_word, word_sets):

    single_cond = F.lit(False)
    if single_word:
        single_pattern = r"(?i)\b(" + "|".join(single_word) + r")\b"
        single_cond = col(desc_col).rlike(single_pattern)
        
    sets_cond = F.lit(False)
    for ws in word_sets:
        ws_cond = F.lit(True)
        for w in ws:
            ws_cond = ws_cond & col(desc_col).rlike(fr"(?i)\b{w}\b")
        sets_cond = sets_cond | ws_cond

    exclude_words = ["history", "due", "following", "infant", "neonate", "dietary"]
    for word in single_word:
        exclude_words.append(f"without {word}")
    exclude_cond = ~col(desc_col).rlike(r"(?i)\b(" + "|".join(exclude_words) + r")\b")

    is_post = (single_cond | sets_cond) & exclude_cond

    return is_post  

def filtering_condition(df, codes, code_prefixes):
    cols = [c for c in df.columns if c.startswith(code_prefixes)]
    condition = reduce(
        lambda a, b: a | b,
        [F.col(c).isin(codes) for c in cols]
    )
    return condition

In [0]:
hypertension_single = ["hypertension"]
hypertension_set = [["high", "blood", "pressure"], ["elevated", "blood", "pressure"]]

hypertension_snomed_codes = snomed_codes_desc.filter(diagnosis_codes("TERM", hypertension_single, hypertension_set)).agg(F.collect_set("CONCEPT_ID")).collect()[0][0]
hypertension_icd10_codes = icd10_codes_desc.filter(diagnosis_codes("DESCRIPTION", hypertension_single, hypertension_set)).agg(F.collect_set("ALT_CODE")).collect()[0][0]

In [0]:
diastolic_blood_pressure_codes = ["271650006", "1091811000000102", "716632005"]
systolic_blood_pressure_codes = ["72313002", "271649006", "716579001", "72313002"]
mean_blood_pressure_codes = ["6797001"]

df_blood_pressure = (
    df_observations
    .filter(col("UCUMUnit") == "mmHg")
    .select("UniqPregID", "ObsDate",  "ObsCode", "ObsValue", "UCUMUnit", "FindingCode", "MasterSnomedCTFindingTerm", "DiagnosisDate")
)

df_obs_hypertension = (
    df_observations
    .withColumn("ObsValue", col("ObsValue").cast("float"))
    .filter(
        (
            (col("UCUMUnit") == "mmHg") &
            ((col("ObsCode").isin(diastolic_blood_pressure_codes) & (col("ObsValue") >= 90))  |     # high diastolic blood pressure
            (col("ObsCode").isin(systolic_blood_pressure_codes) & (col("ObsValue") >= 140)))        # high systolic blood pressure
        ) | (col("FindingCode").isin(hypertension_snomed_codes))                                    # hypertension
    )
    .select("UniqPregID", "DiagnosisDate")
)

hypertension_preg_ids = [row["UniqPregID"] for row in df_obs_hypertension.select("UniqPregID").collect()]

In [0]:
df_msds_hypertension = (
    df_msds_diags
    .filter(col("DiagnosisAndHistory").isin(hypertension_snomed_codes))
    .select("UniqPregID", "DiagnosisDate")
)

hypertension_preg_ids += [row["UniqPregID"] for row in df_msds_hypertension.select("UniqPregID").collect()]

In [0]:
diag_cols = [c for c in df_ecds.columns if c.startswith("DIAGNOSIS_CODE")]

df_ecds_hypertension = (
    df_ecds
    .filter(filtering_condition(df_ecds, hypertension_snomed_codes, ("DIAGNOSIS_CODE")))
    .select("PERSON_ID_DEID", "ARRIVAL_DATE", *diag_cols)
)

df_ecds_hypertension = (
    df_preg_ids
    .join(
        df_ecds_hypertension,
        (
            (df_ecds_hypertension.PERSON_ID_DEID == df_preg_ids.person_id_deid) &
            (df_ecds_hypertension.ARRIVAL_DATE >= df_preg_ids.last_period_date) &
            (df_ecds_hypertension.ARRIVAL_DATE <= df_preg_ids.ld_hosp_disch_date)
        ),
        "inner"
    ).drop(*(["last_period_date", "ld_hosp_disch_date", "PERSON_ID_DEID"] + diag_cols))
    .withColumnRenamed("ARRIVAL_DATE", "DiagnosisDate")
)

hypertension_preg_ids += [row["UniqPregID"] for row in df_ecds_hypertension.select("UniqPregID").collect()]

In [0]:
diag_cols = [c for c in df_hes_apc.columns if c.startswith(("DIAG_3_0", "DIAG_3_1", "DIAG_3_2", "DIAG_4_0", "DIAG_4_1", "DIAG_4_2"))]

df_hes_apc_hypertension = (
    df_hes_apc
    .filter(filtering_condition(df_hes_apc, hypertension_icd10_codes, ("DIAG_3_0", "DIAG_3_1", "DIAG_3_2", "DIAG_4_0", "DIAG_4_1", "DIAG_4_2")))
    .select("PERSON_ID_DEID", "ADMIDATE", *diag_cols)
)

df_hes_apc_hypertension = (
    df_preg_ids
    .join(
        df_hes_apc_hypertension,
        (
            (df_hes_apc_hypertension.PERSON_ID_DEID == df_preg_ids.person_id_deid) &
            (df_hes_apc_hypertension.ADMIDATE >= df_preg_ids.last_period_date) &
            (df_hes_apc_hypertension.ADMIDATE <= df_preg_ids.ld_hosp_disch_date)
        ),
        "inner"
    ).drop(*(["last_period_date", "ld_hosp_disch_date", "PERSON_ID_DEID"] + diag_cols))
    .withColumnRenamed("ADMIDATE", "DiagnosisDate")
)

hypertension_preg_ids += [row["UniqPregID"] for row in df_hes_apc_hypertension.select("UniqPregID").collect()]
hypertension_preg_ids_counts = Counter(hypertension_preg_ids)
mapping_expr = F.create_map(
    *[F.lit(x) for kv in hypertension_preg_ids_counts.items() for x in kv]
)

In [0]:
proteinuria_single = ["proteinuria"]
proteinuria_sets = []

proteinuria_snomed_codes = snomed_codes_desc.filter(diagnosis_codes("TERM", proteinuria_single, proteinuria_sets)).agg(F.collect_set("CONCEPT_ID")).collect()[0][0]
proteinuria_icd10_codes = icd10_codes_desc.filter(diagnosis_codes("DESCRIPTION", proteinuria_single, proteinuria_sets)).agg(F.collect_set("ALT_CODE")).collect()[0][0]

proteinuria_codes = df_observations.filter(diagnosis_codes("MasterSnomedCTFindingTerm", proteinuria_single, proteinuria_sets)).agg(F.collect_set("MasterSnomedCTFindingCode")).collect()[0][0]      # None found in ICD10
proteinuria_codes += proteinuria_snomed_codes

In [0]:
df_obs_proteinuria = (
    df_observations
    .filter(col("FindingCode").isin(proteinuria_codes)) # No observations found with matching term (proteinuria is a finding, not obs)
    .select("UniqPregID", "DiagnosisDate")
)

proteinuria_preg_ids = [row["UniqPregID"] for row in df_obs_proteinuria.select("UniqPregID").collect()]

In [0]:
df_msds_proteinuria = (
    df_msds_diags
    .filter(col("DiagnosisAndHistory").isin(proteinuria_snomed_codes))  # None found in MasterSnomedCTDiagnosisAndHistoryCode
    .select("UniqPregID", "DiagnosisDate")
)

proteinuria_preg_ids += [row["UniqPregID"] for row in df_msds_proteinuria.select("UniqPregID").collect()]

In [0]:
diag_cols = [c for c in df_ecds.columns if c.startswith("DIAGNOSIS_CODE")]

df_ecds_proteinuria = (
    df_ecds
    .filter(filtering_condition(df_ecds, proteinuria_snomed_codes, ("DIAGNOSIS_CODE")))
    .select("PERSON_ID_DEID", "ARRIVAL_DATE", *diag_cols)
)

df_ecds_proteinuria = (
    df_preg_ids
    .join(
        df_ecds_proteinuria,
        (
            (df_ecds_proteinuria.PERSON_ID_DEID == df_preg_ids.person_id_deid) &
            (df_ecds_proteinuria.ARRIVAL_DATE >= df_preg_ids.last_period_date) &
            (df_ecds_proteinuria.ARRIVAL_DATE <= df_preg_ids.ld_hosp_disch_date)
        ),
        "inner"
    ).drop(*(["last_period_date", "ld_hosp_disch_date", "PERSON_ID_DEID"] + diag_cols))
    .withColumnRenamed("ARRIVAL_DATE", "DiagnosisDate")
)

proteinuria_preg_ids += [row["UniqPregID"] for row in df_ecds_proteinuria.select("UniqPregID").collect()]

In [0]:
diag_cols = [c for c in df_hes_apc.columns if c.startswith(("DIAG_3_0", "DIAG_3_1", "DIAG_3_2", "DIAG_4_0", "DIAG_4_1", "DIAG_4_2"))]

df_hes_apc_proteinuria = (
    df_hes_apc
    .filter(filtering_condition(df_hes_apc, proteinuria_icd10_codes, ("DIAG_3_0", "DIAG_3_1", "DIAG_3_2", "DIAG_4_0", "DIAG_4_1", "DIAG_4_2")))
    .select("PERSON_ID_DEID", "ADMIDATE", *diag_cols)
)

df_hes_apc_proteinuria = (
    df_preg_ids
    .join(
        df_hes_apc_proteinuria,
        (
            (df_hes_apc_proteinuria.PERSON_ID_DEID == df_preg_ids.person_id_deid) &
            (df_hes_apc_proteinuria.ADMIDATE >= df_preg_ids.last_period_date) &
            (df_hes_apc_proteinuria.ADMIDATE <= df_preg_ids.ld_hosp_disch_date)
        ),
        "inner"
    ).drop(*(["last_period_date", "ld_hosp_disch_date", "PERSON_ID_DEID"] + diag_cols))
    .withColumnRenamed("ADMIDATE", "DiagnosisDate")
)

proteinuria_preg_ids += [row["UniqPregID"] for row in df_hes_apc_proteinuria.select("UniqPregID").collect()]

In [0]:
PlGF_single = ["sFlt-1", "PlGF", "sFlt-1/PlGF"]
PlGF_set =  [["placental", "growth", "factor"], ["sFlt-1", "PlGF"]]

PlGF_snomed_codes = snomed_codes_desc.filter(diagnosis_codes("TERM", PlGF_single, PlGF_set)).agg(F.collect_set("CONCEPT_ID")).collect()[0][0]    # None found
PlGF_icd10_codes = icd10_codes_desc.filter(diagnosis_codes("DESCRIPTION", PlGF_single, PlGF_set)).agg(F.collect_set("ALT_CODE")).collect()[0][0]
PlGF_codes = df_observations.filter(diagnosis_codes("MasterSnomedCTFindingTerm", PlGF_single, PlGF_set)).agg(F.collect_set("MasterSnomedCTFindingCode")).collect()[0][0]   # None found
PlGF_codes += df_observations.filter(diagnosis_codes("MasterICD10FindingDesc", PlGF_single, PlGF_set)).agg(F.collect_set("MasterICD10FindingCode")).collect()[0][0]     # None found
PlGF_codes += df_observations.filter(diagnosis_codes("MasterSnomedCTObsTerm", PlGF_single, PlGF_set)).agg(F.collect_set("MasterSnomedCTObsCode")).collect()[0][0]       # None found
PlGF_codes += PlGF_snomed_codes
PlGF_codes += ["25761000237100"]            # Manual Snomed search

In [0]:
df_obs_PlGF =  (
    df_observations
    .filter(col("ObsCode").isin(PlGF_codes) | col("FindingCode").isin(PlGF_codes))
    .select("UniqPregID", "DiagnosisDate")
)
#display(df_plfg)   # None found

plgf_preg_ids = [row["UniqPregID"] for row in df_obs_PlGF.select("UniqPregID").collect()]

In [0]:
df_msds_PlGF = (
    df_msds_diags
    .filter(col("DiagnosisAndHistory").isin(PlGF_snomed_codes))
    .select("UniqPregID", "DiagnosisDate")
)

plgf_preg_ids += [row["UniqPregID"] for row in df_msds_PlGF.select("UniqPregID").collect()]

In [0]:
diag_cols = [c for c in df_ecds.columns if c.startswith("DIAGNOSIS_CODE")]

df_ecds_PlGF = (
    df_ecds
    .filter(filtering_condition(df_ecds, PlGF_snomed_codes, ("DIAGNOSIS_CODE")))
    .select("PERSON_ID_DEID", "ARRIVAL_DATE", *diag_cols)
)

df_ecds_PlGF = (
    df_preg_ids
    .join(
        df_ecds_PlGF,
        (
            (df_ecds_PlGF.PERSON_ID_DEID == df_preg_ids.person_id_deid) &
            (df_ecds_PlGF.ARRIVAL_DATE >= df_preg_ids.last_period_date) &
            (df_ecds_PlGF.ARRIVAL_DATE <= df_preg_ids.ld_hosp_disch_date)
        ),
        "inner"
    ).drop(*(["last_period_date", "ld_hosp_disch_date", "PERSON_ID_DEID"] + diag_cols))
    .withColumnRenamed("ARRIVAL_DATE", "DiagnosisDate")
)

plgf_preg_ids += [row["UniqPregID"] for row in df_ecds_PlGF.select("UniqPregID").collect()]

In [0]:
diag_cols = [c for c in df_hes_apc.columns if c.startswith(("DIAG_3_0", "DIAG_3_1", "DIAG_3_2", "DIAG_4_0", "DIAG_4_1", "DIAG_4_2"))]

df_hes_apc_PlGF = (
    df_hes_apc
    .filter(filtering_condition(df_hes_apc, PlGF_icd10_codes, ("DIAG_3_0", "DIAG_3_1", "DIAG_3_2", "DIAG_4_0", "DIAG_4_1", "DIAG_4_2")))
    .select("PERSON_ID_DEID", "ADMIDATE", *diag_cols)
)

df_hes_apc_PlGF = (
    df_preg_ids
    .join(
        df_hes_apc_PlGF,
        (
            (df_hes_apc_PlGF.PERSON_ID_DEID == df_preg_ids.person_id_deid) &
            (df_hes_apc_PlGF.ADMIDATE >= df_preg_ids.last_period_date) &
            (df_hes_apc_PlGF.ADMIDATE <= df_preg_ids.ld_hosp_disch_date)
        ),
        "inner"
    ).drop(*(["last_period_date", "ld_hosp_disch_date", "PERSON_ID_DEID"] + diag_cols))
    .withColumnRenamed("ADMIDATE", "DiagnosisDate")
)

plgf_preg_ids += [row["UniqPregID"] for row in df_hes_apc_PlGF.select("UniqPregID").collect()]

In [0]:
df_hypertension = (
    df_obs_hypertension.unionByName(df_msds_hypertension).unionByName(df_ecds_hypertension).unionByName(df_hes_apc_hypertension)
    .distinct()
)

df_hypertension_counts = (
    df_hypertension
    .groupBy("UniqPregID")
    .count()
    .withColumnRenamed("count", "Hypertension_count")
)

df_proteinuria = (
    df_obs_proteinuria.unionByName(df_msds_proteinuria).unionByName(df_ecds_proteinuria).unionByName(df_hes_apc_proteinuria)
    .distinct()
)

df_proteinuria_counts = (
    df_proteinuria
    .groupBy("UniqPregID")
    .count()
    .withColumnRenamed("count", "Proteinuria_count")
)

df_PlGF = (
    df_obs_PlGF.unionByName(df_msds_PlGF).unionByName(df_ecds_PlGF).unionByName(df_hes_apc_PlGF)
    .distinct()
)

df_PlGF_counts = (
    df_PlGF
    .groupBy("UniqPregID")
    .count()
    .withColumnRenamed("count", "PlGF_count")
)

In [0]:
df_master_final = (
    df_reduced
    .join(df_hypertension_counts, "UniqPregID", "left")
    .join(df_proteinuria_counts, "UniqPregID", "left")
    .join(df_PlGF_counts, "UniqPregID", "left")
    .withColumns({
        "Hypertension_count": F.coalesce(F.col("Hypertension_count"), F.lit(0)),
        "Proteinuria_count": F.coalesce(F.col("Proteinuria_count"), F.lit(0)),
        "PlGF_count": F.coalesce(F.col("PlGF_count"), F.lit(0))
    })
    .withColumns({
        "Hypertension_2x": F.when(F.col("Hypertension_count") >= 2, 1).otherwise(0),
        "Proteinuria_1x": F.when(F.col("Proteinuria_count") >= 1, 1).otherwise(0),
        "PlGF_test": F.when(F.col("PlGF_count") >= 1, 1).otherwise(0)
    })
    .withColumn(
        "Possible_preeclampsia_diagnosis", when(((col("Hypertension_2x") == 1) | (col("Proteinuria_1x") == 1) | (col("PlGF_test") == 1)), 1).otherwise(0)
    )
)

In [0]:
write_table_name = "msds_diag_busy_filtered_final_imputed_reduced_stage"

df_master_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{write_table_name}")